# TrackPulse — Streaming Silver CDF to Historical Silver (Recurring Incremental Job)

**Schedule:** Every 12 hours, after Bronze ingestion and Streaming Silver incremental processing both succeed.

**Prerequisite:** Run `03_bootstrap_historical_silver_once.ipynb` manually once before the recurring ADF trigger. That notebook performs the initial idempotent catch-up and enables Change Data Feed on Streaming Silver.

**Incrementality:** This notebook reads Streaming Silver Change Data Feed and uses its own checkpoint. The first run reads the current Streaming Silver snapshot once; later runs resume from the saved offset and process only subsequent changes. It never reads Historical Silver as a source, never overwrites it, and only inserts event IDs not already present.

**Do not delete/reset** `checkpoints/databricks_streaming/historical_silver_cdf_v1/`. If this path already exists, reuse it.

In [0]:
# Purpose: Configure and validate the incremental CDF-to-Historical-Silver job.
# Safety: Read-only preflight. The existing integration checkpoint is reused and must never be deleted/reset.

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

streaming_silver_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "silver/telemetry_stream/"
)

historical_silver_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "silver/telemetry/"
)

integration_checkpoint_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "checkpoints/databricks_streaming/historical_silver_cdf_v1/"
)

if not DeltaTable.isDeltaTable(spark, streaming_silver_path):
    raise RuntimeError("Streaming Silver is not an existing Delta table.")
if not DeltaTable.isDeltaTable(spark, historical_silver_path):
    raise RuntimeError("Historical Silver is not an existing Delta table.")

source_df = spark.read.format("delta").load(streaming_silver_path)
target_df = spark.read.format("delta").load(historical_silver_path)
source_types = {f.name: f.dataType.simpleString() for f in source_df.schema.fields}
target_types = {f.name: f.dataType.simpleString() for f in target_df.schema.fields}
if source_types != target_types:
    raise ValueError(
        "Streaming Silver and Historical Silver schemas differ. No write was attempted. "
        f"Source schema: {source_types}; target schema: {target_types}"
    )

target_columns = target_df.columns
print("Streaming/Historical schemas match.")
print("CDF checkpoint:", integration_checkpoint_path)
print("The first CDF run reads the current Streaming Silver snapshot once; later runs resume from this checkpoint.")


Streaming/Historical schemas match.
CDF checkpoint: abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/checkpoints/databricks_streaming/historical_silver_cdf_v1/
The first CDF run reads the current Streaming Silver snapshot once; later runs resume from this checkpoint.


In [0]:
# Purpose: Merge only change-feed rows that represent inserts or post-update values into Historical Silver.
# Safety: Historical Silver is append-only by event_id here; matched event IDs are not overwritten.
# CDF's initial snapshot is handled once on the first query start; subsequent ADF runs resume from the checkpoint.

def merge_historical_batch(change_batch_df, batch_id):
    if change_batch_df.limit(1).count() == 0:
        print(f"CDF batch {batch_id}: no changes.")
        return

    eligible_df = change_batch_df.filter(
        F.col("_change_type").isin("insert", "update_postimage")
    )

    if eligible_df.limit(1).count() == 0:
        print(f"CDF batch {batch_id}: no insert/postimage rows to merge.")
        return

    # Select the latest eligible change for each event_id within this micro-batch.
    latest_per_event = Window.partitionBy("event_id").orderBy(
        F.col("_commit_version").desc(),
        F.col("ingested_at").desc_nulls_last()
    )
    merge_source_df = (
        eligible_df
        .filter(F.col("event_id").isNotNull() & (F.trim(F.col("event_id")) != ""))
        .withColumn("_row_number", F.row_number().over(latest_per_event))
        .filter(F.col("_row_number") == 1)
        .select(*target_columns)
    )

    if merge_source_df.limit(1).count() == 0:
        print(f"CDF batch {batch_id}: no rows with valid event_id.")
        return

    (
        DeltaTable.forPath(spark, historical_silver_path)
        .alias("t")
        .merge(merge_source_df.alias("s"), "t.event_id = s.event_id")
        .whenNotMatchedInsertAll()
        .execute()
    )

    operation = (
        DeltaTable.forPath(spark, historical_silver_path)
        .history(1)
        .select("version", "timestamp", "operation", "operationMetrics")
        .first()
    )
    print(f"CDF batch {batch_id} processed. Latest Historical Delta operation: {operation.asDict() if operation else None}")

# Read Delta Change Data Feed instead of rescanning the 10-million-row Historical Silver source.
# No startingVersion is specified: the first run processes the current Streaming Silver snapshot as inserts,
# and the checkpoint makes later runs resume from the last successfully committed source offset.
cdf_stream = (
    spark.readStream
    .format("delta")
    .option("readChangeFeed", "true")
    .load(streaming_silver_path)
)

# Keep inserts and update post-images; ignore pre-images/deletes for this append-only telemetry history.
cdf_stream = cdf_stream.filter(
    F.col("_change_type").isin("insert", "update_postimage")
)

if any(query.isActive for query in spark.streams.active):
    raise RuntimeError("An active streaming query exists. Stop and inspect it before starting this job.")

query = (
    cdf_stream.writeStream
    .foreachBatch(merge_historical_batch)
    .option("checkpointLocation", integration_checkpoint_path)
    .trigger(availableNow=True)
    .start()
)
query.awaitTermination()
print("Streaming Silver CDF to Historical Silver processing finished.")


Streaming Silver CDF to Historical Silver processing finished.


In [0]:
# Purpose: Verify the latest Historical Silver Delta operation and the integration checkpoint.
# Safety: Read-only checks; do not count/reload the full 10-million-row Historical Silver table here.

print("Latest Historical Silver Delta operation:")
display(
    DeltaTable.forPath(spark, historical_silver_path)
    .history(5)
    .select("version", "timestamp", "operation", "operationMetrics")
    .orderBy(F.desc("version"))
)

print("Integration checkpoint entries:")
for item in dbutils.fs.ls(integration_checkpoint_path):
    print(item.name, item.size)


Latest Historical Silver Delta operation:


version,timestamp,operation,operationMetrics
24,2026-10-28T20:03:18.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 1699, materializeSourceTimeMs -> 6, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 0, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 166010, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1670)"
23,2026-10-28T20:02:08.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 7616325, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 2680, materializeSourceTimeMs -> 15, numTargetRowsInserted -> 166010, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 0, numTargetRowsUpdated -> 0, numOutputRows -> 166010, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 166010, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2615)"
22,2026-10-28T15:17:20.000Z,WRITE,"Map(numFiles -> 9, numRemovedFiles -> 9, numRemovedBytes -> 372232597, numDeletionVectorsRemoved -> 0, numOutputRows -> 10000000, numOutputBytes -> 372232615)"
21,2026-10-28T15:04:01.000Z,WRITE,"Map(numFiles -> 9, numRemovedFiles -> 9, numRemovedBytes -> 372232606, numDeletionVectorsRemoved -> 0, numOutputRows -> 10000000, numOutputBytes -> 372232597)"
20,2026-10-28T14:56:06.000Z,WRITE,"Map(numFiles -> 9, numRemovedFiles -> 9, numRemovedBytes -> 372232606, numDeletionVectorsRemoved -> 0, numOutputRows -> 10000000, numOutputBytes -> 372232606)"


Integration checkpoint entries:
__tmp_path_dir/ 0
commits/ 0
metadata 45
offsets/ 0


In [0]:

# Validate row counts, event_id uniqueness, and null IDs across the Delta layers.

paths = {
    "Bronze": "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/bronze/telemetry_stream/",
    "Streaming Silver": "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/silver/telemetry_stream/",
    "Quarantine": "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/silver/quarantine/telemetry_stream/",
    "Historical Silver": "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/silver/telemetry/",
}

for name, path in paths.items():
    df = spark.read.format("delta").load(path)

    total_rows = df.count()
    null_ids = df.filter("event_id IS NULL").count()
    distinct_ids = df.select("event_id").distinct().count()

    print(f"\n{name}")
    print(f"Total rows: {total_rows:,}")
    print(f"Distinct event IDs: {distinct_ids:,}")
    print(f"Null event IDs: {null_ids:,}")
    print(f"Duplicate-ID excess: {total_rows - distinct_ids:,}")



Bronze
Total rows: 167,522
Distinct event IDs: 166,022
Null event IDs: 0
Duplicate-ID excess: 1,500

Streaming Silver
Total rows: 166,010
Distinct event IDs: 166,010
Null event IDs: 0
Duplicate-ID excess: 0

Quarantine
Total rows: 12
Distinct event IDs: 12
Null event IDs: 0
Duplicate-ID excess: 0

Historical Silver
Total rows: 10,166,010
Distinct event IDs: 10,166,010
Null event IDs: 0
Duplicate-ID excess: 0


In [0]:

# Inspect recent Delta Lake commits for Historical Silver.

historical_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "silver/telemetry/"
)

display(
    spark.sql(f"DESCRIBE HISTORY delta.`{historical_path}`")
    .select("version", "timestamp", "operation", "operationMetrics")
    .limit(10)
)


version,timestamp,operation,operationMetrics
24,2026-10-28T20:03:18.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 1699, materializeSourceTimeMs -> 6, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 0, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 166010, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1670)"
23,2026-10-28T20:02:08.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 7616325, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 2680, materializeSourceTimeMs -> 15, numTargetRowsInserted -> 166010, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 0, numTargetRowsUpdated -> 0, numOutputRows -> 166010, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 166010, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2615)"
22,2026-10-28T15:17:20.000Z,WRITE,"Map(numFiles -> 9, numRemovedFiles -> 9, numRemovedBytes -> 372232597, numDeletionVectorsRemoved -> 0, numOutputRows -> 10000000, numOutputBytes -> 372232615)"
21,2026-10-28T15:04:01.000Z,WRITE,"Map(numFiles -> 9, numRemovedFiles -> 9, numRemovedBytes -> 372232606, numDeletionVectorsRemoved -> 0, numOutputRows -> 10000000, numOutputBytes -> 372232597)"
20,2026-10-28T14:56:06.000Z,WRITE,"Map(numFiles -> 9, numRemovedFiles -> 9, numRemovedBytes -> 372232606, numDeletionVectorsRemoved -> 0, numOutputRows -> 10000000, numOutputBytes -> 372232606)"
19,2026-10-28T14:48:27.000Z,WRITE,"Map(numFiles -> 9, numRemovedFiles -> 10, numRemovedBytes -> 373428239, numDeletionVectorsRemoved -> 0, numOutputRows -> 10000000, numOutputBytes -> 372232606)"
18,2026-10-28T11:44:25.000Z,WRITE,"Map(numFiles -> 10, numRemovedFiles -> 10, numRemovedBytes -> 373428239, numDeletionVectorsRemoved -> 0, numOutputRows -> 10000000, numOutputBytes -> 373428239)"
17,2026-10-28T11:35:01.000Z,WRITE,"Map(numFiles -> 10, numRemovedFiles -> 10, numRemovedBytes -> 373428239, numDeletionVectorsRemoved -> 0, numOutputRows -> 10000000, numOutputBytes -> 373428239)"
16,2026-10-28T11:21:16.000Z,WRITE,"Map(numFiles -> 10, numRemovedFiles -> 10, numRemovedBytes -> 373428239, numDeletionVectorsRemoved -> 0, numOutputRows -> 10000000, numOutputBytes -> 373428239)"
15,2026-10-28T10:34:51.000Z,WRITE,"Map(numFiles -> 10, numRemovedFiles -> 10, numRemovedBytes -> 373428239, numDeletionVectorsRemoved -> 0, numOutputRows -> 10000000, numOutputBytes -> 373428239)"


In [0]:

# Verify that every Streaming Silver event_id exists in Historical Silver.

streaming_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "silver/telemetry_stream/"
)

historical_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "silver/telemetry/"
)

streaming_ids = (
    spark.read.format("delta").load(streaming_path)
    .select("event_id").distinct()
)

historical_ids = (
    spark.read.format("delta").load(historical_path)
    .select("event_id").distinct()
)

missing_ids = streaming_ids.join(
    historical_ids,
    on="event_id",
    how="left_anti"
)

missing_count = missing_ids.count()

print(f"Streaming IDs missing from Historical Silver: {missing_count:,}")


Streaming IDs missing from Historical Silver: 0
